In [1]:
from urllib.parse import urlencode, quote

def build_naukri_url(keyword: str, location: str, experience: int, job_age: int = 15) -> str:
    # Format the slug portion so location comes right after "jobs-in"
    formatted_keyword = keyword.strip().lower().replace(" ", "-")
    formatted_location = location.strip().lower().replace(" ", "-")
    
    base_path = f"https://www.naukri.com/{formatted_keyword}-jobs-in-{formatted_location}"
    
    # Keep only experience and jobAge as query parameters
    params = {
        "experience": experience,
        "jobAge": job_age
    }
    
    query_string = urlencode(params, quote_via=quote)
    
    return f"{base_path}?{query_string}"

In [3]:
url = build_naukri_url("FDE", "Hyderabad", 5, 15)

In [17]:
from curl_cffi import requests

def get_naukri_jobs(keywords: str, location: str, page_no: int = 1):
    # Naukri API endpoint
    base_url = "https://www.naukri.com/jobapi/v3/search"
    
    # Query parameters
    params = {
        "noOfResults": 20,
        "urlType": "search_by_keyword",
        "searchType": "adv",
        "keyword": keywords,
        "location": location,
        "pageNo": page_no
    }
    
    # Headers with Appid and Systemid
    headers = {
        "Appid": "109",
        "Systemid": "Naukri",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
        "Clientid": "d3452f1002c385f6",
    }
    
    # Send GET request using curl_cffi impersonating Chrome
    response = requests.get(
        base_url,
        params=params,
        headers=headers,
        impersonate="chrome"
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Request failed with status code {response.status_code}: {response.text}")
        return None



In [23]:
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def extract_naukri_jobs_headless(keyword: str = "fde", location: str = "hyderabad secunderabad"):
    clean_kw = keyword.strip().lower().replace(" ", "-")
    clean_loc = location.strip().lower().replace(" ", "-")
    url = f"https://www.naukri.com/{clean_kw}-jobs-in-{clean_loc}"
    
    # Configure Headless Chrome
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")  # Enable new headless mode
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    # Set standard User-Agent so headless mode isn't detected
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(options=options)
    jobs_list = []
    
    try:
        print(f"Navigating in Headless mode to {url}...")
        driver.get(url)
        
        # Wait up to 15 seconds for job cards to render
        wait = WebDriverWait(driver, 15)
        wait.until(
            EC.presence_of_element_located((
                By.CSS_SELECTOR, 
                "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple"
            ))
        )
        time.sleep(2)
        
        cards = driver.find_elements(By.CSS_SELECTOR, "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple")
        print(f"Headless browser loaded! Found and parsing {len(cards)} job cards...\n")
        
        for card in cards:
            job = {}
            
            # Title & URL
            try:
                title_elem = card.find_element(By.CSS_SELECTOR, "a.title, a.job-title")
                job["title"] = title_elem.text.strip()
                job["url"] = title_elem.get_attribute("href")
            except:
                job["title"], job["url"] = None, None
                
            # Company
            try:
                comp_elem = card.find_element(By.CSS_SELECTOR, "a.comp-name, span.comp-name, .subTitle")
                job["company"] = comp_elem.text.strip()
            except:
                job["company"] = None
                
            # Rating
            try:
                rating_elem = card.find_element(By.CSS_SELECTOR, "span.rating, .rating")
                job["rating"] = rating_elem.text.strip()
            except:
                job["rating"] = None
                
            # Experience
            try:
                exp_elem = card.find_element(By.CSS_SELECTOR, "span.exp-wrap, span.expwdth, .experience")
                job["experience"] = exp_elem.text.strip()
            except:
                job["experience"] = None
                
            # Salary
            try:
                sal_elem = card.find_element(By.CSS_SELECTOR, "span.sal-wrap, span.ni-job-tuple-icon-srp-rupee, .salary")
                job["salary"] = sal_elem.text.strip()
            except:
                job["salary"] = None
                
            # Location
            try:
                loc_elem = card.find_element(By.CSS_SELECTOR, "span.loc-wrap, span.locWdth, .location")
                job["location"] = loc_elem.text.strip()
            except:
                job["location"] = None
                
            # Skills
            try:
                tag_elems = card.find_elements(By.CSS_SELECTOR, "ul.tags-row li, div.tags-gt li, .dot-gt li")
                job["skills"] = [t.text.strip() for t in tag_elems if t.text.strip()]
            except:
                job["skills"] = []
                
            if job["title"]:
                jobs_list.append(job)
                
    except Exception as e:
        print(f"Error in headless browser: {e}")
        
    finally:
        driver.quit()
        
    return jobs_list

# --- Execution ---
if __name__ == "__main__":
    jobs = extract_naukri_jobs_headless(keyword="fde", location="hyderabad secunderabad")
    
    # Print formatted JSON output
    print(json.dumps(jobs, indent=2, ensure_ascii=False))


Navigating in Headless mode to https://www.naukri.com/fde-jobs-in-hyderabad-secunderabad...
Headless browser loaded! Found and parsing 20 job cards...

[
  {
    "title": "Senior FDE Engineer",
    "url": "https://www.naukri.com/job-listings-senior-fde-engineer-milestone-technologies-inc-hyderabad-12-to-14-years-300926501748",
    "company": "Milestone Technologies",
    "rating": "4.0",
    "experience": "12-14 Yrs",
    "salary": null,
    "location": "Hyderabad",
    "skills": []
  },
  {
    "title": "Forward Deployment Engineer (FDE) / Automation Engineer",
    "url": "https://www.naukri.com/job-listings-forward-deployment-engineer-fde-automation-engineer-infosys-limited-hyderabad-7-to-12-years-280926928575",
    "company": "Infosys",
    "rating": "3.5",
    "experience": "7-12 Yrs",
    "salary": null,
    "location": "Hyderabad",
    "skills": []
  },
  {
    "title": "FDE - Agentic Ai Professional",
    "url": "https://www.naukri.com/job-listings-fde-agentic-ai-professional-da